In [27]:
if TARGET in COLS:
    class_dis = train[TARGET].value_counts(normalize=True).round(2)
    print("Distribution of classes",class_dis)

    plt.figure(figsize=(12,4))
    class_dis.sort_values().plot(kind='bar')
else:
    print(f"Column {TARGET} doesnt exist. Make sure it exists")

Column accident_risk doesnt exist. Make sure it exists


In [28]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score ,train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score,mean_squared_error

import xgboost as xgb

TRAIN_PATH = '/kaggle/input/playground-series-s5e10/train.csv'
TEST_PATH = '/kaggle/input/playground-series-s5e10/test.csv'

assert os.path.exists(TRAIN_PATH) , f"This path {TRAIN_Path} doesnt exist. Make sure it exists"
assert os.path.exists(TEST_PATH) , f"This path {TEST_PATH} doesnt exist. Make sure it exists"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train shape:{train.shape}")
print(f"Train shape:{test.shape}")

train.head()

/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv
Train shape:(517754, 14)
Train shape:(172585, 13)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [29]:
TARGET = 'accident_risk'
ID_COL = 'id'

FEATURES_COLS = [c for c in train.columns]
DROP_COLS = ["road_type","time_of_day","id",'accident_risk']
COLS = [c for c in FEATURES_COLS if c not in DROP_COLS ]
print(COLS,"coltest",COLS_TEST)

# Set categorical and numeric cols

CATEGORICAL_COLS = []
NUMERIQUES_COLS = []

for c in COLS:
    if c == ID_COL:
        continue
    if pd.api.types.is_numeric_dtype(train[c]):
        NUMERIQUES_COLS.append(c)
    else:
        CATEGORICAL_COLS.append(c)

print(f"Numeric Columns",NUMERIQUES_COLS)
print(f"Categorical Columns",CATEGORICAL_COLS)

['num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'holiday', 'school_season', 'num_reported_accidents'] coltest ['num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'holiday', 'school_season', 'num_reported_accidents']
Numeric Columns ['num_lanes', 'curvature', 'speed_limit', 'road_signs_present', 'public_road', 'holiday', 'school_season', 'num_reported_accidents']
Categorical Columns ['lighting', 'weather']


In [30]:
numeric_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])

categoric_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore',sparse_output= False))
])

preprocess = ColumnTransformer(
   transformers = [
       ('num',numeric_transformer,NUMERIQUES_COLS),
       ('cat',categoric_transformer,CATEGORICAL_COLS)
   ]
)

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    use_lavel_encoder=False,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocess',preprocess),
    ('model',xgb.XGBRegressor())
]
)

X_train = train[COLS].drop(columns=[DROP_COLS],errors='ignore')
y_train = train[TARGET]

X_test = test[COLS].drop(columns=[DROP_COLS],errors='ignore')


In [31]:
pipeline.fit(X_train,y_train)

y_predict = pipeline.predict(X_test)


In [32]:
print(y_predict)

[0.29179847 0.12265969 0.18947603 ... 0.25436518 0.12598751 0.48809403]


In [33]:
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET: y_predict})


save_path = "submission.csv"
submission.to_csv(save_path, index=False)